# LAB-01 · Kotlin lab: run Kotlin, DSA and Ktor in a notebook

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/LAB-01.ipynb)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: LAB-01
title: "Kotlin lab: run Kotlin, DSA and Ktor in a notebook"
track: lab
date: 2026-10-09
status: in-progress  # not-started | in-progress | partial | done
score: ""
insight: ""
tags: [LAB-01, lab, kotlin, dsa, ktor, colab]
pending:
  - "Run the setup cell in Colab and each example once"
  - "Tell the tutor which cell failed, if any (examples 3 and 4 download libraries)"
```

## How it works

- A cell that starts with `%%kotlin` is compiled and run as a **Kotlin script**: top-level statements run directly, no `main()` needed.
- **Cells do not share state.** Put the function and its checks in the same cell.
- `kotlin.test` is available, so `assertEquals(...)` works for DSA checks.
- Libraries: add `@file:DependsOn("group:artifact:version")` at the top of the cell (needs internet).
- Speed: the first Kotlin cell downloads the compiler (about 80 MB, about a minute). After that each cell takes about 8 seconds.
- Output shows the error and the line in your cell. Add `--timeout 600` after `%%kotlin` for slow cells.

In [ ]:
# Setup: run once per session. Adds the %%kotlin cell magic. Works in Colab and in VS Code / Jupyter.
import os, urllib.request
SRC = next((p for p in ("tools/kotlin_magic.py", "../../tools/kotlin_magic.py") if os.path.exists(p)), None)
if SRC is None:                                   # e.g. Colab: download it from GitHub
    SRC = "kotlin_magic.py"
    urllib.request.urlretrieve("https://raw.githubusercontent.com/BuiltBySwap/engineering-lab/main/tools/kotlin_magic.py", SRC)
%run $SRC

## 1 · Plain Kotlin

In [ ]:
%%kotlin
data class PlanDay(val id: String, val topic: String, val hours: Double)

val days = listOf(
    PlanDay("BE-01", "How a backend works", 1.17),
    PlanDay("AND-01", "Coroutines", 1.17),
    PlanDay("DD-01", "Restart day", 4.33),
)
println("long days: ${days.filter { it.hours > 2 }.map { it.id }}")
println("total hours: ${"%.2f".format(days.sumOf { it.hours })}")

## 2 · A DSA solution with checks (LC 560)

A failing check stops the cell with the line number. Try changing an expected value to see it fail.

In [ ]:
%%kotlin
import kotlin.test.assertEquals

fun subarraySum(nums: IntArray, k: Int): Int {
    var prefixSum = 0
    var count = 0
    val seen = hashMapOf(0 to 1)                      // prefix sum -> times seen so far
    for (number in nums) {
        prefixSum += number
        count += seen[prefixSum - k] ?: 0             // count first...
        seen[prefixSum] = (seen[prefixSum] ?: 0) + 1  // ...then record this prefix
    }
    return count
}

assertEquals(2, subarraySum(intArrayOf(1, 1, 1), 2))
assertEquals(2, subarraySum(intArrayOf(1, 2, 3), 3))
assertEquals(3, subarraySum(intArrayOf(1, -1, 0), 0))
assertEquals(1, subarraySum(intArrayOf(5), 5))
assertEquals(0, subarraySum(intArrayOf(5), 3))
assertEquals(1, subarraySum(intArrayOf(5, -3), 2))    // the sliding-window trap
println("LC 560: all checks passed")

## 3 · Coroutines

⚠️ This cell and the next one download libraries from Maven Central. I could not test them in my own sandbox (no Maven access), so run them once and report any error.

In [ ]:
%%kotlin
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core-jvm:1.10.2")

import kotlinx.coroutines.*
import kotlin.system.measureTimeMillis

runBlocking {
    val withDelay = measureTimeMillis {
        listOf(launch { delay(1000) }, launch { delay(1000) }).joinAll()
    }
    println("two delay(1000)        : $withDelay ms   (the waits overlap)")

    val withSleep = measureTimeMillis {
        listOf(launch { Thread.sleep(1000) }, launch { Thread.sleep(1000) }).joinAll()
    }
    println("two Thread.sleep(1000) : $withSleep ms   (one thread: they queue)")
}

## 4 · Ktor, in-process (experimental)

`testApplication` runs a Ktor app and calls it without opening a port. A server on a port cannot be reached from your browser in Colab, so this is the way to try Ktor here.

In [ ]:
%%kotlin
@file:DependsOn("io.ktor:ktor-server-test-host-jvm:3.6.0")

import io.ktor.client.request.*
import io.ktor.client.statement.*
import io.ktor.http.*
import io.ktor.server.response.*
import io.ktor.server.routing.*
import io.ktor.server.testing.*

// A whole Ktor app inside the test host: no port, no browser, no deployment.
testApplication {
    application {
        routing {
            get("/health") { call.respondText("""{"status":"ok"}""", ContentType.Application.Json) }
            get("/plan/today") { call.respondText("""{"id":"BE-02","topic":"Ktor fundamentals"}""", ContentType.Application.Json) }
        }
    }
    val health = client.get("/health")
    println("GET /health      -> ${health.status}  ${health.bodyAsText()}")
    val today = client.get("/plan/today")
    println("GET /plan/today  -> ${today.status}  ${today.bodyAsText()}")
    println("GET /nope        -> ${client.get("/nope").status}")
}

## Where bigger Kotlin lives

| Code | Where | How to run |
|---|---|---|
| DSA solutions with tests | `kotlin-dsa-kata` | `./gradlew test` |
| The Kaizen server (Ktor) | `kaizen/server-ktor` | `./gradlew run` |
| The Kaizen Android app | `kaizen/android` | Android Studio |

The notebooks teach and show results. The repos are the real, tested code.

## Bug diary

| Error or symptom | Cause | Fix |
|---|---|---|
| | | |